In [2]:
from pathlib import Path
import re 
import numpy as np
import pandas as pd

DATA_PATH = Path("/Users/vakof/Desktop/HOD_car/HOD_car/Preprocessing/audi_q4_secondary_lifecycle.csv")
df = pd.read_csv(DATA_PATH)
df.head(5)


,listing_id,title,subtitle,variant,model_group,transmission,price,currency,price_superscript,is_conditional_price,...,electric_range_city,secondary_scrape_status,secondary_scrape_error,first_seen_timestamp,first_seen_file,last_seen_timestamp,last_seen_file,sold_timestamp,sold_in_file,status
0,39f056dc-fe26-4544-8333-4edabfca1f5b,Audi Q4 e-tron 45,45,Q4 e-tron,Q4 e-tron,Automatic,"46,980",€,1.0,False,...,NaN,ok,NaN,20260423,scrape_audi_q4_20260423_secondary.csv,20260423,scrape_audi_q4_20260423_secondary.csv,20260424.0,scrape_audi_q4_20260424_secondary.csv,sold_or_removed
1,3370c59b-b5f5-4143-86e8-d26e1d9f8eb3,Audi Q4 e-tron Sportback edition S line AHK/He...,Sportback edition S line AHK/Head-up,Q4 e-tron Sportback,Q4 e-tron,Automatic,"54,480",€,1.0,False,...,NaN,ok,NaN,20260423,scrape_audi_q4_20260423_secondary.csv,20260423,scrape_audi_q4_20260423_secondary.csv,20260424.0,scrape_audi_q4_20260424_secondary.csv,sold_or_removed
2,19df05fc-321b-4d18-930c-cefd231c00c8,Audi Q4 e-tron Q4 Sportback 50 e-tron S line M...,Q4 Sportback 50 e-tron S line Matrix GRA Kamera,Q4 e-tron Sportback,Q4 e-tron,Automatic,"31,810",€,1.0,False,...,NaN,ok,NaN,20260423,scrape_audi_q4_20260423_secondary.csv,20260423,scrape_audi_q4_20260423_secondary.csv,20260424.0,scrape_audi_q4_20260424_secondary.csv,sold_or_removed
3,b565d292-f98b-40b9-ba77-771247ecfcc6,Audi Q4 e-tron 45 210 kW,45 210 kW,Q4 e-tron,Q4 e-tron,Automatic,"44,890",€,1.0,False,...,NaN,ok,NaN,20260423,scrape_audi_q4_20260423_secondary.csv,20260423,scrape_audi_q4_20260423_secondary.csv,20260424.0,scrape_audi_q4_20260424_secondary.csv,sold_or_removed
4,21c7def8-abe8-47ca-b194-86d66f3627ce,Audi Q4 e-tron 45 210 kW,45 210 kW,Q4 e-tron,Q4 e-tron,Automatic,"44,890",€,1.0,False,...,NaN,ok,NaN,20260423,scrape_audi_q4_20260423_secondary.csv,20260423,scrape_audi_q4_20260423_secondary.csv,20260424.0,scrape_audi_q4_20260424_secondary.csv,sold_or_removed


In [6]:
df.shape

(2517, 64)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2517 entries, 0 to 2516
Data columns (total 64 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   listing_id                2517 non-null   object 
 1   title                     2517 non-null   object 
 2   subtitle                  2517 non-null   object 
 3   variant                   2504 non-null   object 
 4   model_group               2504 non-null   object 
 5   transmission              2517 non-null   object 
 6   price                     2517 non-null   object 
 7   currency                  2517 non-null   object 
 8   price_superscript         2418 non-null   float64
 9   is_conditional_price      2517 non-null   bool   
 10  mileage                   2517 non-null   object 
 11  first_registration        2517 non-null   object 
 12  fuel_or_powertrain        2517 non-null   object 
 13  power_kw                  2514 non-null   float64
 14  power_hp

In [14]:
df.isna().mean().sort_values(ascending=False).head(30)
df.nunique().sort_values(ascending=False).head(30)

listing_url               2517
primary_image_url         2455
listing_id                2389
subtitle                  1973
title                     1972
mileage                   1821
price                     1278
seller_info_url            462
dealer_id                  462
seller_location            440
zip_code                   435
seller_phone               427
seller_imprint_url         425
street                     423
city                       391
seller_name                348
electric_range             206
seller_rating_count        183
electric_range_city         73
first_registration          60
raw_card_text               55
wltp_consumption            54
image_count                 39
secondary_scrape_error      37
warranty_text               28
last_seen_timestamp         20
first_seen_file             20
last_seen_file              20
first_seen_timestamp        20
sold_in_file                19
dtype: int64

In [3]:
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100,
    "unique_count": df.nunique(),
    "example": df.apply(lambda col: col.dropna().iloc[0] if col.dropna().size > 0 else np.nan)   
    }).sort_values("missing_percentage", ascending=False)
profile

,dtype,missing_count,missing_percentage,unique_count,example
vat_deductible,float64,2517,100.000000,0,NaN
delivery_possible,object,2503,99.443782,1,True
has_360_image,object,2488,98.847835,1,True
secondary_scrape_error,object,2480,98.529996,37,Failed to fetch https://www.autoscout24.com/of...
raw_card_text,object,2460,97.735399,55,"Audi Q4 e-tron Q4 e-tron 40 S line Nav/EPH/20""..."
...,...,...,...,...,...
country_code,object,0,0.000000,1,DE
zip_code,int64,0,0.000000,435,54292
city,object,0,0.000000,391,Trier
image_count,int64,0,0.000000,39,9


In [4]:
clean = df.copy()

In [5]:
def parse_number(series):
    return pd.to_numeric(
        series.astype("string")
        .str.extract(r"([0-9][0-9,.]*)", expand=False)
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

def parse_decimal(series):
    return pd.to_numeric(
        series.astype("string")
        .str.extract(r"([0-9]+(?:\.[0-9]+)?)", expand=False),
        errors="coerce"
    )

In [20]:
clean["price_eur"] = parse_number(clean['price'])
clean['mileage_km']= parse_number(clean['mileage'])
clean['electric_range_km'] = parse_number(clean['electric_range'])
clean['electric_range_city_km'] = parse_number(clean['electric_range_city'])
clean['battery_charging_time_min'] = parse_number(clean['battery_charging_time'])
clean["warranty_months"] = parse_number(clean['warranty_text'])


In [7]:
is_kwh = clean["wltp_consumption"].astype("string").str.contains("kWh/100 km", na=False)
clean["wltp_consumption_kwh_100km"] = np.where(
    is_kwh,
    parse_decimal(clean["wltp_consumption"]),
    np.nan
)

clean["wltp_co2_g_km"] = parse_number(clean["wltp_co2_emissions"])

In [8]:
for col in ["first_seen_timestamp", "last_seen_timestamp", "sold_timestamp"]:
    clean[col + "_dt"] = pd.to_datetime(
        clean[col].astype("Int64").astype("string"),
        format="%Y%m%d",
        errors="coerce"
    )

clean["first_registration_dt"] = pd.to_datetime(
    "01/" + clean["first_registration"].astype("string"),
    format="%d/%m/%Y",
    errors="coerce"
)

In [9]:
clean["sold_flag"] = clean["status"].eq("sold_or_removed")
clean["is_censored"] = clean["status"].eq("still_listed_at_last_scrape")

clean["observed_days"] = (
    clean["last_seen_timestamp_dt"] - clean["first_seen_timestamp_dt"]
).dt.days

clean["days_until_sold_or_removed"] = (
    clean["sold_timestamp_dt"] - clean["first_seen_timestamp_dt"]
).dt.days

latest_scrape_date = clean["last_seen_timestamp_dt"].max()

clean["vehicle_age_months"] = (
    (latest_scrape_date.year - clean["first_registration_dt"].dt.year) * 12
    + (latest_scrape_date.month - clean["first_registration_dt"].dt.month)
)

In [10]:
bool_cols = [
    "is_conditional_price",
    "available_now",
    "warranty_exists",
    "has_full_service_history",
    "had_accident",
    "has_360_image",
    "delivery_possible",
]

def parse_bool(series):
    return series.astype("string").str.lower().map({
        "true": True,
        "false": False
    }).astype("boolean")

for col in bool_cols:
    clean[col + "_clean"] = parse_bool(clean[col])

In [11]:
text = (
    clean["title"].fillna("") + " " +
    clean["subtitle"].fillna("")
).str.lower()

clean["is_sportback"] = clean["variant"].eq("Q4 e-tron Sportback")
clean["is_quattro"] = text.str.contains("quattro", regex=False)
clean["is_s_line"] = text.str.contains("s line|s-line", regex=True)
clean["has_matrix"] = text.str.contains("matrix", regex=False)
clean["has_pano"] = text.str.contains("pano|panorama", regex=True)
clean["has_ahk"] = text.str.contains("ahk", regex=False)
clean["has_hud"] = text.str.contains("hud|head-up|head up", regex=True)
clean["has_acc"] = text.str.contains(r"\bacc\b", regex=True)
clean["has_camera"] = text.str.contains("kamera|camera|rfk", regex=True)


In [12]:
clean["model_number"] = text.str.extract(r"\b(35|40|45|50)\b", expand=False)


In [13]:
clean["duplicate_listing_id"] = clean["listing_id"].duplicated(keep=False)
clean["scrape_error_flag"] = clean["secondary_scrape_status"].eq("error")

clean["suspicious_price"] = clean["price_eur"].lt(15000) | clean["price_eur"].gt(100000)
clean["suspicious_mileage"] = clean["mileage_km"].lt(0) | clean["mileage_km"].gt(250000)
clean["suspicious_wltp"] = clean["wltp_consumption_kwh_100km"].gt(40)
clean["suspicious_co2"] = clean["wltp_co2_g_km"].gt(0)

In [14]:
checks = {
    "price_missing": clean["price_eur"].isna().sum(),
    "mileage_missing": clean["mileage_km"].isna().sum(),
    "bad_date_order": (clean["last_seen_timestamp_dt"] < clean["first_seen_timestamp_dt"]).sum(),
    "sold_without_sold_date": (clean["sold_flag"] & clean["sold_timestamp_dt"].isna()).sum(),
    "still_listed_with_sold_date": (clean["is_censored"] & clean["sold_timestamp_dt"].notna()).sum(),
}

checks

{'price_missing': np.int64(0),
 'mileage_missing': np.int64(3),
 'bad_date_order': np.int64(0),
 'sold_without_sold_date': np.int64(0),
 'still_listed_with_sold_date': np.int64(0)}

In [18]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

clean.to_csv(OUTPUT_DIR / "audi_q4_lifecycle_clean.csv", index=False)
profile.to_csv(OUTPUT_DIR / "audi_q4_column_profile.csv")

In [21]:
clean.to_parquet(OUTPUT_DIR / "audi_q4_lifecycle_clean.parquet", index=False)

In [22]:
ml = clean.copy()

ml["duration_days"] = np.where(
    ml["sold_flag"],
    ml["days_until_sold_or_removed"],
    ml["observed_days"]
)

ml["event_sold_or_removed"] = ml["sold_flag"].astype(int)

for days in [3, 7, 14, 21]:
    sold_within = ml["sold_flag"] & (ml["days_until_sold_or_removed"] <= days)
    eligible = sold_within | (ml["observed_days"] >= days)

    ml[f"eligible_{days}d"] = eligible
    ml[f"sold_within_{days}d"] = np.where(eligible, sold_within.astype(int), np.nan)

In [23]:
text = (
    ml["title"].fillna("") + " " +
    ml["subtitle"].fillna("")
).str.lower()

ml["model_number_v2"] = text.str.extract(
    r"(?:q4\s*)?(35|40|45|50|55)\s*(?:e[- ]?tron|quattro|qu\.|\b)",
    expand=False
)

ml["model_number_v2"] = ml["model_number_v2"].fillna(ml["model_number"])


In [24]:
ml["price_per_kw"] = ml["price_eur"] / ml["power_kw"]
ml["price_per_hp"] = ml["price_eur"] / ml["power_hp"]
ml["price_per_range_km"] = ml["price_eur"] / ml["electric_range_km"]

ml["mileage_per_month"] = ml["mileage_km"] / ml["vehicle_age_months"].replace(0, np.nan)

ml["seller_has_rating"] = ml["seller_rating_stars"].notna().astype(int)
ml["has_warranty_months"] = ml["warranty_months"].notna().astype(int)
ml["has_battery_info"] = ml["battery_ownership"].notna().astype(int)
ml["has_city_range"] = ml["electric_range_city_km"].notna().astype(int)
ml["has_charging_time"] = ml["battery_charging_time_min"].notna().astype(int)

In [25]:
ml.loc[ml["wltp_consumption_kwh_100km"] > 40, "wltp_consumption_kwh_100km"] = np.nan
ml.loc[ml["wltp_co2_g_km"] > 0, "wltp_co2_g_km"] = np.nan
ml.loc[ml["electric_range_km"] < 250, "electric_range_km"] = np.nan
ml.loc[ml["price_eur"] > 90000, "price_eur"] = np.nan

In [26]:
leakage_cols = [
    "price",
    "price_eur",
    "status",
    "sold_flag",
    "is_censored",
    "observed_days",
    "days_until_sold_or_removed",
    "duration_days",
    "event_sold_or_removed",
    "sold_timestamp",
    "sold_timestamp_dt",
    "sold_in_file",
    "last_seen_timestamp",
    "last_seen_timestamp_dt",
    "last_seen_file",
    "secondary_scrape_status",
    "secondary_scrape_error",
    "scrape_error_flag",
]

In [27]:
numeric_features = [
    "mileage_km",
    "vehicle_age_months",
    "power_kw",
    "power_hp",
    "electric_range_km",
    "wltp_consumption_kwh_100km",
    "seller_rating_stars",
    "seller_rating_count",
    "image_count",
    "previous_owner_count",
    "door_count",
    "seat_count",
    "warranty_months",
    "battery_charging_time_min",
    "price_per_kw",
    "price_per_hp",
    "price_per_range_km",
    "mileage_per_month",
]

categorical_features = [
    "variant",
    "model_number_v2",
    "seller_type",
    "body_type",
    "exterior_color",
    "paint_type",
    "interior_color",
    "upholstery_material",
    "city",
]

binary_features = [
    "is_conditional_price_clean",
    "available_now_clean",
    "warranty_exists_clean",
    "has_full_service_history_clean",
    "had_accident_clean",
    "is_sportback",
    "is_quattro",
    "is_s_line",
    "has_matrix",
    "has_pano",
    "has_ahk",
    "has_hud",
    "has_acc",
    "has_camera",
    "duplicate_listing_id",
    "seller_has_rating",
    "has_warranty_months",
    "has_battery_info",
    "has_city_range",
    "has_charging_time",
]

In [28]:
price_model_df = ml[
    numeric_features + categorical_features + binary_features + ["price_eur", "listing_id"]
].copy()

price_model_df = price_model_df.dropna(subset=["price_eur"])

price_model_df.to_parquet(
    OUTPUT_DIR / "audi_q4_price_model_dataset.parquet",
    index=False
)

In [29]:
sale_14d_df = ml[
    numeric_features + categorical_features + binary_features +
    ["price_eur", "sold_within_14d", "eligible_14d", "listing_id"]
].copy()

sale_14d_df = sale_14d_df[sale_14d_df["eligible_14d"]].drop(columns=["eligible_14d"])
sale_14d_df = sale_14d_df.dropna(subset=["sold_within_14d", "price_eur"])

sale_14d_df.to_parquet(
    OUTPUT_DIR / "audi_q4_sale_14d_model_dataset.parquet",
    index=False
)

In [31]:
survival_df = ml[
    numeric_features + categorical_features + binary_features +
    ["price_eur", "duration_days", "event_sold_or_removed", "listing_id"]
].copy()

survival_df = survival_df.dropna(subset=["duration_days", "event_sold_or_removed", "price_eur"])

survival_df.to_parquet(
    OUTPUT_DIR / "audi_q4_survival_model_dataset.parquet",
    index=False
)

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/Users/vakof/Desktop/HOD_car/HOD_car/Preprocessing/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "audi_q4_lifecycle_clean": clean,
    "audi_q4_price_model_dataset": price_model_df,
    "audi_q4_sale_14d_model_dataset": sale_14d_df,
    "audi_q4_survival_model_dataset": survival_df,
}

for name, data in datasets.items():
    csv_path = OUTPUT_DIR / f"{name}.csv"
    parquet_path = OUTPUT_DIR / f"{name}.parquet"

    data.to_csv(csv_path, index=False)
    data.to_parquet(parquet_path, index=False)

    print(f"Saved {name}: {data.shape[0]} rows, {data.shape[1]} columns")
    print("CSV:", csv_path)
    print("Parquet:", parquet_path)

profile_path = OUTPUT_DIR / "audi_q4_column_profile.csv"
profile.to_csv(profile_path, index=True)

print("Profile:", profile_path)
print("Done.")